# Compare mean power around RA-AP

This notebook analyzes the mean wavelet power in specified frequency bands around RA-APs.

## Initialization - don't change

In [ ]:
import sys

sys.path.append('..')

from pathlib import Path

import pandas as pd

from param_classes.workflows.wavelet_analysis import WindowedWaveletAnalysisParams
from param_classes.workflows.input_output import InputDataParams
from workflows.run_util.io import collect_paths_from_table

data_params = InputDataParams()
wavelet_params = WindowedWaveletAnalysisParams()

## Color settings

Set the colors that will be used for coloring the markers on the strip plot and the cellwise average firing frequency curve.

In [ ]:
palette = {
    'cell_type': {
        'PYR_L2-3': '#803300ff',
        'PYR_L4-6': '#502d16ff',
        'RSI_L2-3': '#ff48a3ff',
        'RSI_L4-6': '#800080ff',
        'FSI_L2-3': '#d54500ff',
        'FSI_L4-6': '#7a0000ff'
    },
    'spike_type': {
        'AIS-AP': "#3150B7",
        'RA-AP': '#00ff00',
        'UNKNOWN': '#000000b2'
    }
}

## Input-output settings

Base directory paths
 - `MAT_DIR` - base dir of the raw **mat** files, grouped into subfolders
 - `PROCESSED_DIR` - base dir of the preprocessed data and results

In [ ]:
MAT_DIR = 'C:\\Datasets\\raap\\invivo'
PROCESSED_DIR = 'C:\\tmp\\Bozso2025\\PutRAAP\\Results\\New'

In [ ]:
input_param = {
    'mode': 'table',
    'table_path': f'{PROCESSED_DIR}\\data_invivo.xlsx',
    'select_rules': [
        {'selected': 1}
    ]
}

In [ ]:
data_params.raw_data.path= MAT_DIR
data_params.raw_data.extension='mat'
data_params.spike_features.path = f'{PROCESSED_DIR}\\SpikeFeatures'
data_params.spike_features.extension = 'csv'

Set the output base directory path. All results of this analysis (cellwise aggregated tables, grouped results, figures, statistics) is going to saved there.

In [ ]:
output_path = Path(PROCESSED_DIR) / 'Peri_RaAp_power_comparison'
result_path = output_path / '01_results'
individual_path = result_path / 'individual'
figure_path = output_path / '02_figures'
statistics_path = output_path / '03_statistics'

## Analysis parameters

Name of the spike types

In [ ]:
spike_types = ['AIS-AP', 'UNKNOWN', 'RA-AP']

List of the groups in the order of display.

In [ ]:
group_list = [
    'FSI_L2-3',
    'RSI_L2-3',
    'PYR_L2-3',
    'FSI_L4-6',
    'RSI_L4-6',
    'PYR_L4-6'
]

 - `wavelet_params.cut_window`: window size to cut, relative to the spike

In [ ]:
wavelet_params.window_size = 1.0

 - `wavelet_params.signal.name`: signal name to analyze
 - `wavelet_params.signal.fs`: resampled sampling rate of the signal

In [ ]:
wavelet_params.signal.name = 'wideband'
wavelet_params.signal.fs = 5000

 - `wavelet_params.wavelet.low`: lowest analyzed frequency
 - `wavelet_params.wavelet.high`: highest analyzed frequency
 - `wavelet_params.wavelet.step`: step size between analyzed frequencies
 - `wavelet_params.wavelet.n_cycles`: number of cycles in the wavelet
 - `wavelet_params.wavelet.length`: length of the wavelet

In [ ]:
wavelet_params.wavelet.low = 1.0
wavelet_params.wavelet.high = 30.0
wavelet_params.wavelet.step = 1.0
wavelet_params.wavelet.n_cycles = 7
wavelet_params.wavelet.length = 1.0

In [ ]:
wavelet_params.clim = (None, None)

 - `episode_name`: The name of the episode to analyze

In [ ]:
frequency_bands = [
    (1, 4),
    (5, 10),
    (12, 18)
]
time_intervals = [
    (-wavelet_params.window_size, 0),
    (0, wavelet_params.window_size)
]

## Plot parameters

 - `y_limit`: y axis limit for the boxplots

In [ ]:
y_limit = (-2, 2)

## Run analysis

Collect input files according to the table.

In [ ]:
input_files = collect_paths_from_table(data_params, input_param)
n_cells = len(input_files)
n_freq_bands = len(frequency_bands)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from workflows.peri_raap_band_power_analysis import (
    calculate_mean_band_power_for_cell
)

power_dfs = [
    calculate_mean_band_power_for_cell(
        input_file,
        params=wavelet_params,
        frequency_bands=frequency_bands,
        time_intervals=time_intervals
    ) for input_file in input_files
]
power_dff = pd.concat(power_dfs)

In [ ]:
# import numpy as np
# from scipy.stats import ranksums
# from statsmodels.stats.multitest import multipletests

# stat_results = []
# for band_i, band in enumerate(frequency_bands):
#     p_raws = []
#     groups_medians  = []
#     for group_name in group_list:
#         group_values = []
#         for spike_type in ['AIS-AP', 'RA-AP']:
#             group_values.append(flatten_frame[
#                 (flatten_frame['part_id'] == part_i) &
#                 (flatten_frame['group'] == group_name) &
#                 (flatten_frame['spike_type'] == spike_type)
#             ]['value'].to_numpy())
#         _, p = ranksums(group_values[0], group_values[1])
#         groups_medians.append(
#             (np.median(group_values[0]), np.median(group_values[1]))
#         )
#         p_raws.append(p)

#     _, p_adjs, _, _ = multipletests(p_raws, method='holm')

#     for group_name, group_medians, p_raw, p_adj in zip(group_list, groups_medians, p_raws, p_adjs):
#         stat_results.append({
#             'part_id': part_i,
#             'group': group_name,
#             'AIS-AP_median': group_medians[0],
#             'RA-AP_median': group_medians[1],
#             'p_raw': p_raw,
#             'p_adj': p_adj
#         })
# stat_result_df = pd.DataFrame(stat_results)
# statistics_path.mkdir(exist_ok=True, parents=True)
# stat_result_df.to_excel(
#     f'{statistics_path}/spike_type_in_{episode_name}_part_comparison.xlsx',
#     index=False
# )

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

from workflows.spike_ac_analysis import save_figure


part_labels = [
    '_'.join([str(time) for time in time_interval])
    for time_interval in time_intervals
]

figure_path.mkdir(exist_ok=True, parents=True)

for band_i, band in enumerate(frequency_bands):
    band_label = '_'.join([str(freq) for freq in band])

    fig, axes = plt.subplots(1, len(group_list), figsize=(20, 5))
    for col_i, group_name in enumerate(group_list):
        ax = axes[col_i]
        group_band_data = power_dff[
            (power_dff['group'] == group_name) &
            (power_dff['band'] == band_label)
        ]

        sns.boxplot(
            data=group_band_data,
            x='part',
            y='power',
            width=0.5,
            fill=False,
            color='gray',
            fliersize=0,
            ax=ax
        )
        for part_label in part_labels:
            color = palette['cell_type'][group_name]
            group_band_part_data = group_band_data[group_band_data['part'] == part_label]
            sns.stripplot(
                data=group_band_part_data,
                x='part',
                y='power',
                color=color,
                ax=ax
            )

        ax.set_title(f'{group_name}')
        ax.set_ylim(y_limit)
    plt.suptitle(f'[{band[0]}, {band[1]}] Hz')
    save_figure(
        figure_path,
        fig,
        f'mean_power_around_raap_in_{band_label}_Hz_boxplot',
        to_svg=True
    )
    plt.close(fig)
